

Introducción al Modelado Espacial de Robots


Resumen



In [1]:


syms Tij(x_i_j,y_i_j,z_i_j,gi_j,bi_j,ai_j)

% Homogeneous transform
Tij(x_i_j,y_i_j,z_i_j,gi_j,bi_j,ai_j) = [cos(ai_j)*cos(bi_j) cos(ai_j)*sin(bi_j)*sin(gi_j)-sin(ai_j)*cos(gi_j) sin(ai_j)*sin(gi_j)+cos(ai_j)*sin(bi_j)*cos(gi_j) x_i_j; sin(ai_j)*cos(bi_j) cos(ai_j)*cos(gi_j)+sin(ai_j)*sin(bi_j)*sin(gi_j) sin(ai_j)*sin(bi_j)*cos(gi_j)-cos(ai_j)*sin(gi_j) y_i_j; -sin(bi_j) cos(bi_j)*sin(gi_j) cos(bi_j)*cos(gi_j) z_i_j; 0 0 0 1]


## Modelado de la posición del robot 3R


In [2]:
syms z_O_1 z_1_2 z_2_3 z_3_P  %parametros
syms theta_O_1 theta_1_2 theta_2_3 %grados de libertad

T_O_1 = Tij(0,0,z_O_1,0,0,theta_O_1)

In [3]:
T_1_2 = Tij(0,0,z_1_2,0,theta_1_2,0)

In [4]:
T_2_3 = Tij(0,0,z_2_3,0,theta_2_3,0)

In [5]:
T_3_P = Tij(0,0,z_3_P,0,0,0)

In [6]:

T_O_P = simplify(T_O_1*T_1_2*T_2_3*T_3_P)


Vector de postura



In [7]:
xi_O_P = [T_O_P(1,4); T_O_P(2,4); T_O_P(3,4); T_O_P(1,1); T_O_P(2,2); T_O_P(3,3)]

In [8]:

p_O_P = [T_O_P(1,4); T_O_P(2,4); T_O_P(3,4)]


## Modelo cinemático directo de las velocidades

Calculo del Jacobino


In [9]:

%J_theta = simplify(jacobian(xi_O_P,[theta_O_1,theta_1_2, theta_2_3]))
J_theta = simplify(jacobian(p_O_P, [theta_O_1, theta_1_2, theta_2_3]));

In [10]:
det(J_theta)

### Modelo cinemático inverso de las velocidades


In [11]:
%J_theta_psin = pinv(J_theta)




Modelo dinámico del robot en el espacio


Calculo de los centros de masa de los eslabones


In [12]:
syms theta_dot_O_1 theta_dot_1_2 theta_dot_2_3 z_1_C1 z_2_C2 z_3_C3

T_1_C1 = Tij(0,0,z_1_C1,0,0,0)

In [13]:

T_O_C1 = simplify(T_O_1*T_1_C1);

p_O_C1 = [T_O_C1(1,4); T_O_C1(2,4); T_O_C1(3,4)]

In [14]:

v_O_C1 = diff(p_O_C1, theta_O_1)*theta_dot_O_1

In [15]:
%

T_2_C2 = Tij(0,0,z_2_C2,0,0,0)

In [16]:

T_O_C2 = simplify(T_O_1*T_1_2*T_2_C2);

p_O_C2 = [T_O_C2(1,4); T_O_C2(2,4); T_O_C2(3,4)]

In [17]:

v_O_C2 = diff(p_O_C2, theta_O_1)*theta_dot_O_1 + diff(p_O_C2, theta_1_2)*theta_dot_1_2

In [18]:

%

T_3_C3 = Tij(0,0,z_3_C3,0,0,0)

In [19]:

T_O_C3 = simplify(T_O_1*T_1_2*T_2_3*T_3_C3);

p_O_C3 = [T_O_C3(1,4); T_O_C3(2,4); T_O_C3(3,4)]

In [20]:

v_O_C3 = diff(p_O_C3, theta_O_1)*theta_dot_O_1 + diff(p_O_C3, theta_1_2)*theta_dot_1_2 + diff(p_O_C3, theta_2_3)*theta_dot_2_3



Calculo de la propagación de las velocidades angulares



In [21]:
% Matrices de Rotación

R_O_1 = [T_O_1(1,1),T_O_1(1,2),T_O_1(2,3); T_O_1(2,1),T_O_1(1,2),T_O_1(2,3);T_O_1(3,1),T_O_1(3,2),T_O_1(3,3)];
R_1_O = transpose(R_O_1)

In [22]:

R_1_2 = [T_1_2(1,1),T_1_2(1,2),T_1_2(2,3); T_1_2(2,1),T_1_2(1,2),T_1_2(2,3);T_1_2(3,1),T_1_2(3,2),T_1_2(3,3)];
R_2_1 = transpose(R_1_2)

In [23]:

R_2_3 = [T_2_3(1,1),T_2_3(1,2),T_2_3(2,3); T_2_3(2,1),T_2_3(1,2),T_2_3(2,3);T_2_3(3,1),T_2_3(3,2),T_2_3(3,3)];
R_3_2 = transpose(R_2_3)

In [24]:

% calculo de las velocidades angulares
omega_O_O = [0;0;0];
nu_1 = [0;0;1];
omega_1_1 = R_O_1*omega_O_O + nu_1*theta_dot_O_1

In [25]:

nu_2 = [0;1;0];
omega_2_2 = R_2_1*omega_1_1 + nu_2*theta_dot_1_2

In [26]:

nu_3 = [0;1;0];
omega_3_3 = R_3_2*omega_2_2 + nu_3*theta_dot_2_3


Matrices de inercia



In [27]:
syms I_xx1 I_yy1 I_zz1 I_xx2 I_yy2 I_zz2 I_xx3 I_yy3 I_zz3

I_C1 = [I_xx1, 0, 0; 0, I_yy1, 0; 0, 0, I_zz1]

In [28]:
I_C2 = [I_xx2, 0, 0; 0, I_yy2, 0; 0, 0, I_zz2]

In [29]:
I_C3 = [I_xx3, 0, 0; 0, I_yy3, 0; 0, 0, I_zz3]


Calculo del Lagrangeano



In [30]:
syms m_1 m_2 m_3 g

% energía cinetica
k_1 = 1/2*m_1*transpose(v_O_C1)*v_O_C1 + transpose(omega_1_1)*I_C1*omega_1_1

In [31]:
k_2 = 1/2*m_2*transpose(v_O_C2)*v_O_C2 + transpose(omega_2_2)*I_C2*omega_2_2

In [32]:
k_3 = 1/2*m_3*transpose(v_O_C3)*v_O_C3 + transpose(omega_3_3)*I_C1*omega_3_3

In [33]:
% energía potencial
g_O = [0;0;-g];

u_1 = -m_1*transpose(p_O_C1)*g_O

In [34]:
u_2 = -m_2*transpose(p_O_C2)*g_O

In [35]:
u_3 = -m_3*transpose(p_O_C3)*g_O

In [36]:
% lagrangeano

L = (k_1+k_2+k_3)-(u_1+u_2+u_3)


Calculo de los pares


In [37]:
syms theta_ddot_O_1 theta_ddot_1_2 theta_ddot_2_3

D_1 = diff(L,theta_dot_O_1)

In [38]:

tao_1 = (diff(D_1,theta_O_1)*theta_dot_O_1 + diff( ...
    D_1,theta_1_2)*theta_dot_1_2 + diff( ...
    D_1,theta_2_3)*theta_dot_2_3 + diff( ...
    D_1,theta_dot_O_1)*theta_ddot_O_1 + diff( ...
    D_1,theta_dot_1_2)*theta_ddot_1_2 + diff( ...
    D_1,theta_dot_2_3)*theta_ddot_2_3) - diff(L,theta_O_1)

In [39]:

D_2 = diff(L,theta_dot_1_2)

In [40]:
tao_2 = (diff(D_2,theta_O_1)*theta_dot_O_1 + diff( ...
    D_2,theta_1_2)*theta_dot_1_2 + diff( ...
    D_2,theta_2_3)*theta_dot_2_3 + diff( ...
    D_2,theta_dot_O_1)*theta_ddot_O_1 + diff( ...
    D_2,theta_dot_1_2)*theta_ddot_1_2 + diff( ...
    D_2,theta_dot_2_3)*theta_ddot_2_3) - diff(L,theta_1_2)

In [41]:

D_3 = diff(L,theta_dot_2_3)

In [42]:
tao_3 = (diff(D_3,theta_O_1)*theta_dot_O_1 + diff( ...
    D_3,theta_1_2)*theta_dot_1_2 + diff( ...
    D_3,theta_2_3)*theta_dot_2_3 + diff( ...
    D_3,theta_dot_O_1)*theta_ddot_O_1 + diff( ...
    D_3,theta_dot_1_2)*theta_ddot_1_2 + diff( ...
    D_3,theta_dot_2_3)*theta_ddot_2_3) - diff(L,theta_2_3)

In [44]:
% Vector de par

tau = collect([tao_1; tao_2; tao_3],[m_1,m_2,m_3, theta_ddot_O_1,theta_ddot_1_2,theta_ddot_2_3])

In [45]:

M_1 = subs(tau,[theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3,theta_dot_O_1, theta_dot_1_2, theta_dot_2_3,g],[1, 0, 0,0, 0, 0,0])

In [46]:
M_2 = subs(tau,[theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3,theta_dot_O_1, theta_dot_1_2, theta_dot_2_3,g],[0, 1, 0,0, 0, 0,0])

In [47]:
M_3 = subs(tau,[theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3,theta_dot_O_1, theta_dot_1_2, theta_dot_2_3,g],[0, 0, 1,0, 0, 0,0])

In [48]:

M_q = collect([M_1 M_2 M_3],[m_1,m_2,m_3])

In [49]:

V_q = subs(tau,[theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3,theta_dot_O_1, theta_dot_1_2, theta_dot_2_3,g],[0, 0, 0,theta_dot_O_1, theta_dot_1_2, theta_dot_2_3,0])

In [50]:
G_q = subs(tau,[theta_ddot_O_1, theta_ddot_1_2, theta_ddot_2_3,theta_dot_O_1, theta_dot_1_2, theta_dot_2_3,g],[0, 0, 0,0, 0, 0,g])

## Declaración de Valores del robot de 3GDL

In [51]:
%% 1. Primero declaras todos los valores numéricos
m1_val = 0.214;
m2_val = 0.448;
m3_val = 0.304;

z_O_1_val = 0.085;
z_1_2_val = 0.039;
z_2_3_val = 0.234;
z_3_P_val = 0.284;

z_1_C1_val = 0.067155;
z_2_C2_val = 0.081682;
z_3_C3_val = 0.083936;

I_xx1_val = 0.000068;
I_yy1_val = 0.000166;
I_zz1_val = 0.000137;

I_xx2_val = 0.002115;
I_yy2_val = 0.002187;
I_zz2_val = 0.00016;

I_xx3_val = 0.002431;
I_yy3_val = 0.002447;
I_zz3_val = 0.000052;

g_val = 9.81;

%% 2. Luego los valores de articulaciones
theta_O_1_val = 0;
theta_1_2_val = 0;
theta_2_3_val = 0;
theta_dot_O_1_val = 0;
theta_dot_1_2_val = 0;
theta_dot_2_3_val = 0;

%% 3. Y al final la sustitución
params_sym = [m_1, m_2, m_3, ...
              z_O_1, z_1_2, z_2_3, z_3_P, ...
              z_1_C1, z_2_C2, z_3_C3, ...
              I_xx1, I_yy1, I_zz1, ...
              I_xx2, I_yy2, I_zz2, ...
              I_xx3, I_yy3, I_zz3, g, ...
              theta_O_1, theta_1_2, theta_2_3];

params_val = [m1_val, m2_val, m3_val, ...
              z_O_1_val, z_1_2_val, z_2_3_val, z_3_P_val, ...
              z_1_C1_val, z_2_C2_val, z_3_C3_val, ...
              I_xx1_val, I_yy1_val, I_zz1_val, ...
              I_xx2_val, I_yy2_val, I_zz2_val, ...
              I_xx3_val, I_yy3_val, I_zz3_val, g_val, ...
              theta_O_1_val, theta_1_2_val, theta_2_3_val];

params_sym_v = [params_sym, ...
                theta_dot_O_1, theta_dot_1_2, theta_dot_2_3];

params_val_v = [params_val, ...
                theta_dot_O_1_val, theta_dot_1_2_val, theta_dot_2_3_val];

M_q_num = double(subs(M_q, params_sym, params_val))

M_q_num = 3x3
    0.0009         0         0
         0    0.0381    0.0081
         0    0.0081    0.0025

In [52]:
V_q_num = double(subs(V_q, params_sym_v, params_val_v))

V_q_num = 3x1
     0
     0
     0

In [53]:
G_q_num = double(subs(G_q, params_sym, params_val))

G_q_num = 3x1
     0
     0
     0


## Sustiución física

In [54]:
%% Sustitución solo de parámetros físicos (sin thetas)
params_sym_fis = [m_1, m_2, m_3, ...
                  z_O_1, z_1_2, z_2_3, z_3_P, ...
                  z_1_C1, z_2_C2, z_3_C3, ...
                  I_xx1, I_yy1, I_zz1, ...
                  I_xx2, I_yy2, I_zz2, ...
                  I_xx3, I_yy3, I_zz3, g];

params_val_fis = [m1_val, m2_val, m3_val, ...
                  z_O_1_val, z_1_2_val, z_2_3_val, z_3_P_val, ...
                  z_1_C1_val, z_2_C2_val, z_3_C3_val, ...
                  I_xx1_val, I_yy1_val, I_zz1_val, ...
                  I_xx2_val, I_yy2_val, I_zz2_val, ...
                  I_xx3_val, I_yy3_val, I_zz3_val, g_val];

M_q_sim = subs(M_q, params_sym_fis, params_val_fis)

In [55]:
V_q_sim = subs(V_q, params_sym_fis, params_val_fis)

In [56]:
G_q_sim = subs(G_q, params_sym_fis, params_val_fis)

## Sustitución numérica completa

In [57]:
%% Sustitución de parámetros físicos (thetas quedan simbólicas)
params_sym_fis = [m_1, m_2, m_3, ...
                  z_O_1, z_1_2, z_2_3, z_3_P, ...
                  z_1_C1, z_2_C2, z_3_C3, ...
                  I_xx1, I_yy1, I_zz1, ...
                  I_xx2, I_yy2, I_zz2, ...
                  I_xx3, I_yy3, I_zz3, g];

params_val_fis = [m1_val, m2_val, m3_val, ...
                  z_O_1_val, z_1_2_val, z_2_3_val, z_3_P_val, ...
                  z_1_C1_val, z_2_C2_val, z_3_C3_val, ...
                  I_xx1_val, I_yy1_val, I_zz1_val, ...
                  I_xx2_val, I_yy2_val, I_zz2_val, ...
                  I_xx3_val, I_yy3_val, I_zz3_val, g_val];

M_q_sim = simplify(subs(M_q, params_sym_fis, params_val_fis))

In [58]:
V_q_sim = simplify(subs(V_q, params_sym_fis, params_val_fis))

In [59]:
G_q_sim = simplify(subs(G_q, params_sym_fis, params_val_fis))